# Resampling and Frequency Conversion

**Resampling** = converting a time series from one frequency to another.

| Direction | Name | Example |
|---|---|---|
| High freq → low freq (aggregating many points into fewer) | **Downsampling** | daily → monthly |
| Low freq → high freq (no aggregation, just more/emptier bins) | **Upsampling** | weekly → daily |
| Same "size," different alignment | **Neither** | `W-WED` → `W-FRI` |

> Not every frequency conversion is strictly "up" or "down." Converting `W-WED` (weekly, anchored on Wednesday) to `W-FRI` doesn't add or remove data points — it just shifts which day of the week each weekly bucket ends on.

**Mental model:** `resample` behaves like `groupby`, except the "groups" are time intervals instead of column values. The pattern is always:

```python
series_or_frame.resample(rule).agg_function()
```

1. `resample(rule)` buckets the index into time intervals (like `groupby` buckets rows by key).
2. The aggregate call that follows (`.mean()`, `.sum()`, `.ohlc()`, `.ffill()`, `.asfreq()`, ...) decides what happens to each bucket.

In [1]:
import pandas as pd 
import numpy as np 

In [2]:
dates = pd.date_range("2000-01-01", periods=100)

ts = pd.Series(np.random.standard_normal(len(dates)), index=dates)

ts

2000-01-01    0.911563
2000-01-02   -0.721046
2000-01-03    1.731857
2000-01-04    0.507444
2000-01-05   -1.774649
                ...   
2000-04-05   -0.617614
2000-04-06   -0.549289
2000-04-07   -1.525703
2000-04-08   -1.535149
2000-04-09    1.660683
Freq: D, Length: 100, dtype: float64

In [3]:
ts.resample("ME").mean()

2000-01-31   -0.019182
2000-02-29    0.209697
2000-03-31    0.035104
2000-04-30   -0.263533
Freq: ME, dtype: float64

In [4]:
ts.resample("ME").mean().to_period("M")

2000-01   -0.019182
2000-02    0.209697
2000-03    0.035104
2000-04   -0.263533
Freq: M, dtype: float64

`resample` is a flexible method that can be used to process large time series.

### Table 11-5. Resample Method Arguments (pandas 2.x)

| Argument     | Description                                                                                                   |
|--------------|-----------------------------------------------------------------------------------------------------------------|
| `rule`       | String, DateOffset, or timedelta indicating the desired resampled frequency (e.g., `"M"`, `"5min"`, `Second(15)`) |
| `axis`       | **Deprecated.** Resample along this axis; default `axis=0`. Removed in future versions — resample on index only |
| `closed`     | For downsampling, which end of each interval is closed (inclusive): `"right"` or `"left"` (default varies by freq) |
| `label`      | For downsampling, how to label each aggregated bin: with the `"right"` or `"left"` bin edge (default varies by freq) |
| `on`         | Resample on a column instead of the index; column must be datetime-like                                        |
| `level`      | Resample on a specific level of a MultiIndex; level must be datetime-like                                       |
| `origin`     | The "base" timestamp from which to determine the resampling bin edges (e.g., `"epoch"`, `"start"`, `"start_day"`, `"end"`, `"end_day"`, or a Timestamp) |
| `offset`     | An offset timedelta added to `origin`; alternative to `origin` for shifting bin edges                          |
| `group_keys` | Whether to include the group keys in the result when resampling produces a grouped object                      |

### Removed / Deprecated Arguments (no longer used this way)

| Old Argument       | Status                                                                                     |
|---------------------|---------------------------------------------------------------------------------------------|
| `fill_method`        | Removed. Do the fill explicitly after resampling: `.resample(...).mean().ffill()`          |
| `kind`               | Deprecated. Do the conversion explicitly: `.resample(...).mean().to_period()` or `.to_timestamp()` |
| `loffset`            | Removed. Shift the resulting index explicitly: `result.index = result.index + offset`      |
| `convention`         | Deprecated (only applied to period resampling). Convert with `.to_timestamp(how=...)` before/after instead |

## Downsampling

**Downsampling** = aggregating data down to a regular, *lower* frequency.

- The input data doesn't need to already be at a fixed frequency — `resample` just needs *some* datetime index to slice up.
- The frequency you pass (`rule`) defines **bin edges** that chop the time series into intervals, which are then aggregated. For example, resampling to `"ME"` (month end) carves the data into one-month intervals.
- Each interval is *half-open*: one edge is included, the other excluded. That guarantees every timestamp falls into exactly one bin, and the bins together cover the whole time range with no gaps or overlaps.

Two settings control how those intervals are built, and they're easy to confuse:

| Setting | Controls |
|---|---|
| `closed` | Which edge of the interval is *inclusive* — `"left"` or `"right"` |
| `label` | Which edge is used to *name* the resulting bin — `"left"` or `"right"` |

> ⚠️ **These are independent.** You can close on the left but label with the right edge, or any other combination — the examples below walk through each case, and a summary table follows them.

**Let's look at some one-minute frequency data:**

In [5]:
dates = pd.date_range("2000-01-01", periods=12, freq="min")

ts = pd.Series(np.arange(len(dates)), index=dates)

ts

2000-01-01 00:00:00     0
2000-01-01 00:01:00     1
2000-01-01 00:02:00     2
2000-01-01 00:03:00     3
2000-01-01 00:04:00     4
2000-01-01 00:05:00     5
2000-01-01 00:06:00     6
2000-01-01 00:07:00     7
2000-01-01 00:08:00     8
2000-01-01 00:09:00     9
2000-01-01 00:10:00    10
2000-01-01 00:11:00    11
Freq: min, dtype: int64

Suppose you wanted to aggregate this data into five-minute chunks or *bars* by taking the sum of each group:

In [6]:
ts.resample("5min").sum()

2000-01-01 00:00:00    10
2000-01-01 00:05:00    35
2000-01-01 00:10:00    21
Freq: 5min, dtype: int64

By default, `closed="left"` for sub-daily frequencies like `"5min"` — the **left** edge of each interval is inclusive. So `00:00` belongs to the `[00:00, 00:05)` interval, and `00:05` is excluded from it (it starts the *next* bin instead). That's why the first result above (`10 = 0+1+2+3+4`) only sums minutes `00:00`–`00:04`, and `00:05` (value `5`) rolls into the second bin.

In [7]:
ts.resample("5min", closed="right").sum()

1999-12-31 23:55:00     0
2000-01-01 00:00:00    15
2000-01-01 00:05:00    40
2000-01-01 00:10:00    11
Freq: 5min, dtype: int64

The resulting time series is labeled by the timestamps from the left side of each bin. By passing `label="right"`, you can label them with the right bin edge:

In [8]:
ts.resample("5min", closed="right", label="right").sum()

2000-01-01 00:00:00     0
2000-01-01 00:05:00    15
2000-01-01 00:10:00    40
2000-01-01 00:15:00    11
Freq: 5min, dtype: int64

### The four `closed` × `label` combinations

Using the same 5-minute buckets as above:

| `closed` | `label` | Which values get summed into the "00:00" bucket | Result label |
|---|---|---|---|
| `"left"` (default here) | `"left"` (default here) | `00:00`–`00:04` | `00:00` |
| `"left"` | `"right"` | `00:00`–`00:04` | `00:05` |
| `"right"` | `"left"` | `23:55`–`23:59` (previous day) | `23:55` |
| `"right"` | `"right"` (used above) | `23:55`–`23:59` (previous day) | `00:00` |

> ⚠️ **The default isn't always `"left"`.** For `"ME"`, `"YE"`, `"QE"`/`"Q"`, `"BM"`, and `"W"`-family frequencies, pandas defaults to `closed="right", label="right"` instead — those all describe "as of the end of the period," so closing/labeling on the right edge matches the intuition. Sub-daily and daily frequencies (`"D"`, `"h"`, `"min"`, `"s"`, ...) default to `closed="left", label="left"`. That's why `ts.resample("ME").mean()` earlier in this notebook "just worked" without passing `closed`/`label` — the defaults already matched what you'd expect for a month-end aggregate.

**Rule of thumb:** if a resampled result looks shifted by one bucket from what you expected, it's almost always a `closed`/`label` mismatch — check this table before assuming the aggregation itself is wrong.

Lastly, you might want to shift the result index by some amount, say subtracting one second from the right edge to make it more clear which interval the timestamp refers to. To do this, add an offset to the resulting index:

In [9]:
from pandas.tseries.frequencies import to_offset

result = ts.resample("5min", closed="right", label="right").sum()

result.index = result.index + to_offset("-1s")

result

1999-12-31 23:59:59     0
2000-01-01 00:04:59    15
2000-01-01 00:09:59    40
2000-01-01 00:14:59    11
Freq: 5min, dtype: int64

### Open-High-Low-Close (OHLC) resampling

A common way to summarize a bucket of price data (candlestick charts in finance) is four numbers per bucket:

- **open** — first value in the bucket
- **high** — max value in the bucket
- **low** — min value in the bucket
- **close** — last value in the bucket

`.resample(...).ohlc()` computes all four in a single pass and returns a DataFrame with those four columns — instead of one aggregated value the way `.mean()` or `.sum()` would.

> Even outside finance, OHLC is a handy way to downsample without throwing away the shape of the data inside each bucket — you keep the extremes and endpoints instead of collapsing everything to a single average.

In [10]:
ts = pd.Series(np.random.permutation(np.arange(len(dates))), index=dates)

ts.resample("5min").ohlc()

,open,high,low,close
2000-01-01 00:00:00,5,11,2,2
2000-01-01 00:05:00,3,9,0,8
2000-01-01 00:10:00,1,4,1,4


## Upsampling and Interpolation

**Upsampling** converts from a *lower* frequency to a *higher* one — going from fewer bins to more bins. There's nothing to aggregate (each new bin can have at most one existing value in it), so the real question becomes: **what goes in the new, empty slots?**

Let's consider a DataFrame with some weekly data:

In [11]:
frame = pd.DataFrame(np.random.standard_normal((2, 4)), index=pd.date_range("2000-01-01", 
                    periods = 2, freq = "W-WED"),
                    columns=["Colorado", "Texas", "New York", "Ohio"])

frame

,Colorado,Texas,New York,Ohio
2000-01-05,0.401864,-0.238098,0.631281,1.794241
2000-01-12,-0.027338,0.753099,-0.098727,0.928979


Because each week only has one row, resampling to daily (`"D"`) doesn't aggregate anything — it just creates 7x as many rows as there are data points. The new rows with no corresponding weekly value become `NaN`. `.asfreq()` performs exactly this reindex-only conversion: change the frequency, add no fill logic.

In [12]:
df_daily = frame.resample("D").asfreq()

df_daily

,Colorado,Texas,New York,Ohio
2000-01-05,0.401864,-0.238098,0.631281,1.794241
2000-01-06,NaN,NaN,NaN,NaN
2000-01-07,NaN,NaN,NaN,NaN
2000-01-08,NaN,NaN,NaN,NaN
2000-01-09,NaN,NaN,NaN,NaN
2000-01-10,NaN,NaN,NaN,NaN
2000-01-11,NaN,NaN,NaN,NaN
2000-01-12,-0.027338,0.753099,-0.098727,0.928979


Suppose you wanted to fill forward each weekly value on the non-Wednesday. The same filling or interpolation methods available in the `fillna` and `reindex` methods are available for resampling:

In [13]:
frame.resample("D").ffill()

,Colorado,Texas,New York,Ohio
2000-01-05,0.401864,-0.238098,0.631281,1.794241
2000-01-06,0.401864,-0.238098,0.631281,1.794241
2000-01-07,0.401864,-0.238098,0.631281,1.794241
2000-01-08,0.401864,-0.238098,0.631281,1.794241
2000-01-09,0.401864,-0.238098,0.631281,1.794241
2000-01-10,0.401864,-0.238098,0.631281,1.794241
2000-01-11,0.401864,-0.238098,0.631281,1.794241
2000-01-12,-0.027338,0.753099,-0.098727,0.928979


You can similarly choose to only fill a certain number of periods forward to limit how far to continue using an observed value:

In [14]:
frame.resample("D").ffill(limit=2)

,Colorado,Texas,New York,Ohio
2000-01-05,0.401864,-0.238098,0.631281,1.794241
2000-01-06,0.401864,-0.238098,0.631281,1.794241
2000-01-07,0.401864,-0.238098,0.631281,1.794241
2000-01-08,NaN,NaN,NaN,NaN
2000-01-09,NaN,NaN,NaN,NaN
2000-01-10,NaN,NaN,NaN,NaN
2000-01-11,NaN,NaN,NaN,NaN
2000-01-12,-0.027338,0.753099,-0.098727,0.928979


> **Typo fix:** the original note here read "the new date index need to coincide with the old one at all," which is missing a "not" and says the opposite of what's true. The correct statement is: **the new date index does *not* need to coincide with the old one at all.**

The example below resamples `W-WED` data (weeks ending Wednesday) using the rule `"W-THU"` (weeks ending Thursday) — a completely different weekly anchor than the source data. pandas still figures out, for each new Thursday-anchored date, which old Wednesday-anchored value to forward-fill from:

In [15]:
frame.resample("W-THU").ffill()

,Colorado,Texas,New York,Ohio
2000-01-06,0.401864,-0.238098,0.631281,1.794241
2000-01-13,-0.027338,0.753099,-0.098727,0.928979


## Resampling with Periods

Resampling data indexed by `PeriodIndex` works similarly to `DatetimeIndex`, with one conceptual difference worth keeping in mind: a **period represents a whole span of time** (e.g., the entire month of `"2000-01"`), not a single instant. That distinction is what drives the `convention` behavior and the stricter up/downsampling rules below.

In [16]:
frame = pd.DataFrame(np.random.standard_normal((24, 4)),
                    index=pd.period_range("1-2000", "12-2001", freq="M"),
                    columns=["Colorado", "Texas", "New York", "Ohio"])

frame.head()

,Colorado,Texas,New York,Ohio
2000-01,2.514384,0.424600,0.825398,-0.997186
2000-02,-0.608181,0.360248,-1.470499,-0.067603
2000-03,-0.568386,0.231933,-0.442316,1.205709
2000-04,-0.309415,-0.506769,-0.546962,1.534431
2000-05,1.246575,-0.030173,-1.606858,-1.725837


In [17]:
annual_frame = frame.resample("Y-DEC").mean()

annual_frame

,Colorado,Texas,New York,Ohio
2000,0.557285,0.097675,-0.029864,-0.050876
2001,0.101032,-0.007475,0.103808,0.088808


Upsampling periods is more nuanced than upsampling timestamps: because each period is a *span*, pandas has to decide **where inside the new, higher-frequency spans to place the value** — at the start of the span, or the end. That's what `convention` controls:

- `convention="start"` (default) — place the annual value at the **first** sub-period (e.g., `Q1`)
- `convention="end"` — place the annual value at the **last** sub-period (e.g., `Q4`)

Compare the two cells below: the first (`ffill`, default `convention="start"`) starts filling from `2000Q1` onward. The second (`asfreq`, `convention="end"`) places the single value at `2000Q4` instead, leaving the surrounding quarters as `NaN` since `asfreq` does no filling.

In [18]:
# Q-DEC: Quarterly, year ending in December
annual_frame.resample("Q-DEC").ffill()

,Colorado,Texas,New York,Ohio
2000Q1,0.557285,0.097675,-0.029864,-0.050876
2000Q2,0.557285,0.097675,-0.029864,-0.050876
2000Q3,0.557285,0.097675,-0.029864,-0.050876
2000Q4,0.557285,0.097675,-0.029864,-0.050876
2001Q1,0.101032,-0.007475,0.103808,0.088808
2001Q2,0.101032,-0.007475,0.103808,0.088808
2001Q3,0.101032,-0.007475,0.103808,0.088808
2001Q4,0.101032,-0.007475,0.103808,0.088808


In [19]:
annual_frame.resample("Q-DEC", convention="end").asfreq()

,Colorado,Texas,New York,Ohio
2000Q4,0.557285,0.097675,-0.029864,-0.050876
2001Q1,NaN,NaN,NaN,NaN
2001Q2,NaN,NaN,NaN,NaN
2001Q3,NaN,NaN,NaN,NaN
2001Q4,0.101032,-0.007475,0.103808,0.088808


Since periods refer to time *spans* rather than instants, pandas enforces stricter rules about which conversions are even valid:

- **Downsampling:** the target frequency must be a *subperiod* of the source frequency.
- **Upsampling:** the target frequency must be a *superperiod* of the source frequency.

If the two frequencies' calendars don't nest cleanly, pandas raises an exception. This mostly bites you with fiscal-year-anchored quarterly, annual, and weekly frequencies, since their period boundaries depend on which month the "year" starts in.

**Why the example below works:** `annual_frame` is `Y-DEC` (calendar year, ends December). Quarterly frequencies that share the same *set* of quarter-end months as `Y-DEC` — namely `Q-MAR`, `Q-JUN`, `Q-SEP`, and `Q-DEC` (all four end their quarters in March/June/September/December, just with a different quarter labeled "Q1") — nest cleanly under it. That's why `annual_frame.resample("Q-MAR")` below succeeds. A frequency like `Q-JAN` (quarter-ends in Jan/Apr/Jul/Oct) would *not* share a month with `Y-DEC` and would raise instead.

In [20]:
annual_frame.resample("Q-MAR").ffill()

,Colorado,Texas,New York,Ohio
2000Q4,0.557285,0.097675,-0.029864,-0.050876
2001Q1,0.557285,0.097675,-0.029864,-0.050876
2001Q2,0.557285,0.097675,-0.029864,-0.050876
2001Q3,0.557285,0.097675,-0.029864,-0.050876
2001Q4,0.101032,-0.007475,0.103808,0.088808
2002Q1,0.101032,-0.007475,0.103808,0.088808
2002Q2,0.101032,-0.007475,0.103808,0.088808
2002Q3,0.101032,-0.007475,0.103808,0.088808


## Grouped Time Resampling

Conceptually, `resample` is just `groupby` where the groups happen to be time buckets instead of column values. Here's a small example table:

In [21]:
N = 15

times = pd.date_range("2017-05-20 00:00", freq="1min", periods=N)

df = pd.DataFrame({"time": times, "value": np.arange(N)})

df

,time,value
0,2017-05-20 00:00:00,0
1,2017-05-20 00:01:00,1
2,2017-05-20 00:02:00,2
3,2017-05-20 00:03:00,3
4,2017-05-20 00:04:00,4
5,2017-05-20 00:05:00,5
6,2017-05-20 00:06:00,6
7,2017-05-20 00:07:00,7
8,2017-05-20 00:08:00,8
9,2017-05-20 00:09:00,9


Here, we can index by `"time` and then resample:

In [22]:
df.set_index("time").resample("5min").count()

,value
time,
2017-05-20 00:00:00,5
2017-05-20 00:05:00,5
2017-05-20 00:10:00,5


Suppose a DataFrame contains multiple time series, marked by an additional group key column:

In [23]:
df2 = pd.DataFrame({"time": times.repeat(3), 
                    "key":np.tile(["a", "b", "c"], N), 
                    "value": np.arange(N * 3.)})

df2.head(7)

,time,key,value
0,2017-05-20 00:00:00,a,0.0
1,2017-05-20 00:00:00,b,1.0
2,2017-05-20 00:00:00,c,2.0
3,2017-05-20 00:01:00,a,3.0
4,2017-05-20 00:01:00,b,4.0
5,2017-05-20 00:01:00,c,5.0
6,2017-05-20 00:02:00,a,6.0


The DataFrame above is in **long format**: instead of one column per series (`Colorado`, `Texas`, ...), every series is stacked into a single `value` column and distinguished by a `key` column. Plain `df.resample(...)` doesn't help here — there's no single per-group index to resample, since we need to bucket by time **and** by `key` at the same time.

`pandas.Grouper` lets you drop a "resample rule" straight into a regular `groupby` call, combining a time-based grouping with an ordinary column-based one:

In [24]:
time_key = pd.Grouper(freq="5min")

We can then set the time index, group by `"key"` and `time_key`, and aggregate:

In [25]:
resampled = (df2.set_index("time").groupby(["key", time_key]).sum())

resampled

value
key time                      
a   2017-05-20 00:00:00   30.0
    2017-05-20 00:05:00  105.0
    2017-05-20 00:10:00  180.0
b   2017-05-20 00:00:00   35.0
    2017-05-20 00:05:00  110.0
    2017-05-20 00:10:00  185.0
c   2017-05-20 00:00:00   40.0
    2017-05-20 00:05:00  115.0
    2017-05-20 00:10:00  190.0

**Constraint:** `pandas.Grouper(freq=...)` requires the time column to be the index — that's why `.set_index("time")` is called before `.groupby(...)` above. (If you don't want to set the index, `pd.Grouper(key="time", freq=...)` works the same way without requiring a datetime index — it points `Grouper` at a specific column instead.)

## Recap

| Task | How |
|---|---|
| Downsample (aggregate to a lower freq) | `.resample(rule).mean()` / `.sum()` / `.ohlc()` / ... |
| Upsample, no fill (leave gaps as `NaN`) | `.resample(rule).asfreq()` |
| Upsample, forward-fill | `.resample(rule).ffill(limit=...)` |
| Change which edge is inclusive | `closed="left"` / `closed="right"` |
| Change which edge labels the bin | `label="left"` / `label="right"` |
| Choose where a period's value lands when upsampling a `PeriodIndex` | `convention="start"` / `convention="end"` |
| Resample per group in a long-format table | `df.set_index(time_col).groupby([group_col, pd.Grouper(freq=rule)])` |

**Things that tend to trip people up:**
- `closed`/`label` defaults flip depending on frequency — right-aligned for `M`/`Y`/`Q`/`W`-family frequencies, left-aligned for everything else (see the callout earlier in this notebook).
- Upsampling doesn't aggregate anything — it just creates new, empty rows that you then have to explicitly fill (`asfreq`, `ffill`, `bfill`, `interpolate`).
- With a `PeriodIndex`, frequencies must nest cleanly (subperiod/superperiod) or pandas raises an exception — this mostly matters for fiscal-year-anchored quarterly/annual/weekly frequencies.
- `pandas.Grouper(freq=...)` needs a datetime **index**, or an explicit `key=` pointing at a datetime column — it can't infer which column to bucket by on its own.